# Lung Cancer CNN Classification Pipeline

This notebook provides a team-friendly training and testing pipeline for lung cancer image classification using PyTorch. All teammates run identical code with reproducible results.

**For Teams:**
- Install dependencies once in the shared venv
- Modify data paths and hyperparameters in the Config cell
- Run training and testing interactively
- Share results as JSON/CSV exports

In [ ]:
import sys
import json
import os
from pathlib import Path

# Check Python version
print(f"Python version: {sys.version}")

# Import core dependencies
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration & Paths

All teammates must update these paths and hyperparameters in this single cell.

In [ ]:
# ===== SHARED CONFIGURATION =====
# All teammates modify this section and commit to version control

# Paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_PATH = OUTPUT_DIR / "best_model.pt"
REPORT_PATH = OUTPUT_DIR / "evaluation_report.json"

# Hyperparameters
CONFIG = {
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 4,
    "epochs": 15,
    "learning_rate": 1e-4,
    "val_split": 0.15,
    "seed": 42,
    "pretrained": True,
    "freeze_backbone": False,
}

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

# Validate that data directory exists
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}\nCreate it or update DATA_DIR path.")

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Config: {json.dumps(CONFIG, indent=2)}")

## Import Shared Model & Utilities

Load the reusable model class and training utilities from `model.py`. This keeps the notebook clean and shared business logic in `.py` files.

In [ ]:
%load_ext autoreload
%autoreload 2

# Import model from model.py
from model import LungCancerCNN

def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")

## Utility Functions

Define reusable functions for data loading, training, and evaluation. These reduce code duplication across cells.

In [ ]:
def build_transforms(image_size: int):
    """Create train and eval data augmentation pipelines."""
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_transform, eval_transform

def make_dataloaders(data_dir: Path, image_size: int, batch_size: int, num_workers: int, val_split: float, seed: int):
    """Create train, val, test dataloaders."""
    train_transform, eval_transform = build_transforms(image_size)
    
    train_dir = data_dir / "train"
    val_dir = data_dir / "val"
    test_dir = data_dir / "test"
    
    if train_dir.exists() and test_dir.exists():
        full_train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
        if val_dir.exists():
            train_dataset = full_train_dataset
            val_dataset = datasets.ImageFolder(val_dir, transform=eval_transform)
        else:
            val_size = max(1, int(val_split * len(full_train_dataset)))
            train_size = len(full_train_dataset) - val_size
            generator = torch.Generator().manual_seed(seed)
            train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)
            val_dataset.dataset = datasets.ImageFolder(train_dir, transform=eval_transform)
        test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)
    else:
        full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
        test_size = max(1, int(0.15 * len(full_dataset)))
        val_size = max(1, int(val_split * len(full_dataset)))
        train_size = len(full_dataset) - val_size - test_size
        generator = torch.Generator().manual_seed(seed)
        train_dataset, val_dataset, test_dataset = random_split(
            full_dataset, [train_size, val_size, test_size], generator=generator
        )
        val_dataset.dataset = datasets.ImageFolder(data_dir, transform=eval_transform)
        test_dataset.dataset = datasets.ImageFolder(data_dir, transform=eval_transform)
    
    class_to_idx = train_dataset.dataset.class_to_idx if isinstance(train_dataset, torch.utils.data.Subset) else train_dataset.class_to_idx
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    
    return train_loader, val_loader, test_loader, class_to_idx

def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    """Train for one epoch."""
    model.train()
    total_loss, total_acc, count = 0.0, 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            outputs = model(images)
            loss = criterion(outputs, labels)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        total_acc += (torch.argmax(outputs, dim=1) == labels).float().mean().item()
        count += 1
    return total_loss / max(1, count), total_acc / max(1, count)

def eval_epoch(model, loader, criterion, device):
    """Evaluate for one epoch."""
    model.eval()
    total_loss, total_acc, count = 0.0, 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            total_acc += (torch.argmax(outputs, dim=1) == labels).float().mean().item()
            count += 1
    return total_loss / max(1, count), total_acc / max(1, count)

print("✓ Utility functions loaded")

## Load Data & Initialize Model

In [ ]:
train_loader, val_loader, test_loader, class_to_idx = make_dataloaders(
    DATA_DIR, CONFIG["image_size"], CONFIG["batch_size"], CONFIG["num_workers"], CONFIG["val_split"], CONFIG["seed"]
)

idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

print(f"✓ Classes: {list(class_to_idx.keys())}")
print(f"✓ Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# Initialize model
model = LungCancerCNN(num_classes=num_classes, pretrained=CONFIG["pretrained"], freeze_backbone=CONFIG["freeze_backbone"]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG["learning_rate"])
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"✓ Model initialized on {device}")
print(f"✓ Pretrained: {CONFIG['pretrained']}, Freeze backbone: {CONFIG['freeze_backbone']}")

## Train Model

In [ ]:
best_val_acc = 0.0
best_epoch = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        # Save checkpoint
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "class_to_idx": class_to_idx,
            "config": CONFIG,
            "val_acc": val_acc,
        }, CHECKPOINT_PATH)
        print(f"   → Best model saved to {CHECKPOINT_PATH}")

print(f"\n✓ Training complete. Best epoch: {best_epoch} (val_acc={best_val_acc:.4f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train", marker="o")
ax1.plot(history["val_loss"], label="Val", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], label="Train", marker="o")
ax2.plot(history["val_acc"], label="Val", marker="s")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Training & Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Training history saved to {OUTPUT_DIR / 'training_history.png'}")

## Load Best Model & Evaluate on Test Set

In [ ]:
# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"✓ Best model loaded from epoch {checkpoint['epoch']}")

# Evaluate on test set
y_true = []
y_pred = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1).cpu().numpy()
        y_pred.extend(predictions.tolist())
        y_true.extend(labels.numpy().tolist())

y_true_names = [idx_to_class[idx] for idx in y_true]
y_pred_names = [idx_to_class[idx] for idx in y_pred]

test_accuracy = accuracy_score(y_true_names, y_pred_names)
print(f"\n✓ Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_true_names, y_pred_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true_names, y_pred_names, labels=list(class_to_idx.keys()))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_to_idx.keys(), yticklabels=class_to_idx.keys())
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix (Test Accuracy: {test_accuracy:.4f})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Confusion matrix saved to {OUTPUT_DIR / 'confusion_matrix.png'}")

## Export Results

In [ ]:
report_dict = classification_report(y_true_names, y_pred_names, output_dict=True)

report = {
    "accuracy": test_accuracy,
    "best_epoch": best_epoch,
    "best_val_acc": best_val_acc,
    "classification_report": report_dict,
    "confusion_matrix": cm.tolist(),
    "classes": class_to_idx,
    "config": CONFIG,
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"✓ Evaluation report saved to {REPORT_PATH}")
print(f"\nSummary:")
print(f"  - Best epoch: {best_epoch} (val_acc={best_val_acc:.4f})")
print(f"  - Test accuracy: {test_accuracy:.4f}")
print(f"  - Artifacts: {OUTPUT_DIR}")

## Team Collaboration Guide

**For teammates:**

1. **Environment**: All code runs in `.venv/` virtual environment. Install dependencies once: `pip install -r requirements.txt`

2. **Configuration**: Edit the **Configuration cell** to adjust hyperparameters, paths, and seeds. Commit changes to version control.

3. **Autoreload**: The notebook uses `%autoreload 2`, so changes to `model.py` are reloaded automatically without restarting the kernel.

4. **Versioning**: Commit `.ipynb` to git. For cleaner diffs, use `Jupytext` to pair `.ipynb` with `.py` files:
   ```bash
   pip install jupytext
   jupytext --to py lung_cancer_pipeline.ipynb  # Creates .py version
   ```

5. **Testing**: Add pytest-based tests in a `tests/` folder and run them in a dedicated cell to catch regressions.

6. **Artifacts**: Results (models, plots, JSON reports) are saved in `outputs/`. Share these without committing large files (use git-lfs if needed).

7. **Reproducibility**: Always run cells from top to bottom. Use the **Execute Notebook End-to-End** cell to validate the entire pipeline before sharing.